[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anandjha90/Advanced-Python-for-GenAI-Prep-Training/blob/main/notebooks/04_Machine_Learning.ipynb)

# Module 4: Machine Learning (40 min)

Machine Learning (ML) is the science of letting computers learn patterns from data instead of following hand-written rules. Given examples of inputs and outputs, an ML algorithm finds a function that maps inputs to outputs and generalises to new, unseen data.

## 4.1 Types of machine learning

![Figure 6: The three families of machine learning with business examples.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d06_ml_types.png)
*Figure 6: The three families of machine learning with business examples.*


- **Supervised learning**: data has labels (the right answer). **Regression** predicts a number (delivery time, sales); **classification** predicts a category (returned or not, spam or not).
- **Unsupervised learning**: no labels. The algorithm finds structure on its own: **clustering** (customer segments) and **dimensionality reduction** (PCA).
- **Reinforcement learning**: an agent takes actions and learns from rewards. RLHF (Reinforcement Learning from Human Feedback) is how chat LLMs are aligned with human preferences.

Key vocabulary: **features** (X) are the input columns, the **target** or **label** (y) is what we predict, a **model** is the learned function, **training** is fitting the model to data, and **inference** is using the model on new data.

## 4.2 The ML workflow

![Figure 7: The end-to-end machine learning workflow. Data preparation usually takes most of the effort.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d07_ml_workflow.png)
*Figure 7: The end-to-end machine learning workflow. Data preparation usually takes most of the effort.*


1. Define the business problem and success metric. 2. Collect data. 3. Clean data and explore it (EDA). 4. Engineer features. 5. Split into train and test sets. 6. Train several models. 7. Evaluate and tune. 8. Deploy and monitor for drift.

## 4.3 Data preprocessing

Models need clean, numeric input on comparable scales.

- **Missing values**: impute with median (numeric, robust to outliers) or most frequent value (categorical).
- **Encoding**: **one-hot encoding** turns a categorical column into 0/1 columns, one per category.
- **Scaling**: **StandardScaler** converts a column to mean 0, std 1. Distance-based and gradient-based models (KNN, SVM, logistic regression, neural networks) need it; tree models do not.
- **Train/test split**: hold back 20 to 30% of data to measure performance on unseen data. Use `stratify=y` for classification so both sets have the same class balance.
- **Data leakage**: never fit imputers or scalers on the test set. A `Pipeline` with a `ColumnTransformer` guarantees that preprocessing is learned from training data only.

> **Real-world example**
>
> A churn dataset has missing ages, a missing device value and text columns (city tier, device). The pipeline below fills, encodes and scales them in a repeatable way, so the exact same steps run in training and in production.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Real-world example: raw customer data with missing values and text categories
df = pd.DataFrame({
    "age":       [25, 41, np.nan, 19, 35, 52, 28, np.nan, 44, 31],
    "city_tier": ["Tier1", "Tier2", "Tier1", "Tier3", "Tier1",
                  "Tier2", "Tier3", "Tier1", "Tier2", "Tier1"],
    "device":    ["App", "Web", "App", np.nan, "Desktop", "App", "Web", "App", "Desktop", "App"],
    "orders":    [3, 12, 7, 1, 9, 15, 2, 5, 11, 6],
    "avg_spend": [1200, 3400, 2100, 600, 2800, 4100, 750, 1500, 3000, 1900],
    "churned":   [1, 0, 0, 1, 0, 0, 1, 1, 0, 0],
})
print("Missing values per column:\n", df.isna().sum())

X = df.drop(columns="churned")
y = df["churned"]

# 1) Split FIRST so the test set never influences preprocessing (avoids data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 2) Different treatment for numeric and categorical columns
numeric_cols = ["age", "orders", "avg_spend"]
categorical_cols = ["city_tier", "device"]

numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler())])
categorical_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                             ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([("num", numeric_pipe, numeric_cols),
                                  ("cat", categorical_pipe, categorical_cols)])

# 3) fit on train only, then transform both
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print("\nFeatures after preprocessing:", list(feature_names))
print(pd.DataFrame(X_train_ready, columns=feature_names).round(2).to_string())
print("\nTrain shape:", X_train_ready.shape, "| Test shape:", X_test_ready.shape)

## 4.4 Regression

Regression predicts a continuous number.

- **Linear regression** fits y = w1x1 + w2x2 + ... + b by minimising the squared error. It is simple and interpretable: each coefficient is the change in y for a one-unit change in that feature, holding the others constant.
- **Random Forest regression** averages many decision trees and captures non-linear patterns and interactions.

| Metric | Formula idea | Interpretation |
|---|---|---|
| MAE | Average of absolute errors | Average miss, in the target's units |
| RMSE | Square root of average squared error | Like MAE but punishes big misses more |
| R-squared | 1 - (model error / variance of y) | Share of variation explained (1 = perfect, 0 = no better than the mean) |


> **Real-world example**
>
> Property portals estimate house prices from income, location and house features. We use the real California Housing dataset (20,640 districts). Random Forest explains about 80% of price variation versus about 58% for linear regression, because location effects are non-linear.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Real dataset: 20,640 California districts. Target = median house value (in $100,000s)
data = fetch_california_housing(as_frame=True)
df = data.frame
print(df.head())

X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"{name:18} MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}")
    return pred

lin_pred = evaluate("Linear Regression", LinearRegression())
rf_pred = evaluate("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))

# Interpreting linear regression: each coefficient = change in price per unit of the feature
coefs = pd.Series(LinearRegression().fit(X_train, y_train).coef_, index=X.columns).sort_values()
print("\nLinear regression coefficients:\n", coefs.round(3))

plt.figure(figsize=(6, 6))
plt.scatter(y_test, rf_pred, s=4, alpha=0.3)
plt.plot([0, 5], [0, 5], "r--", label="Perfect prediction")
plt.xlabel("Actual value"); plt.ylabel("Predicted value")
plt.title("Random Forest: actual vs predicted"); plt.legend()
plt.show()

## 4.5 Classification and evaluation metrics

Classification predicts a category. Popular algorithms:

- **Logistic regression**: a linear model passed through the sigmoid function to output a probability. Fast, interpretable, strong baseline.
- **Decision tree**: a sequence of if-else splits learned from data. Easy to explain but overfits easily.
- **Random Forest**: many trees on random samples of rows and features, with a majority vote. Robust and accurate.
- **Gradient boosting**: trees built one after another, each fixing the previous errors (XGBoost, LightGBM). Often the best on tabular data.

![Figure 8: Underfitting, good fit and overfitting (left three panels); the confusion matrix and its metrics (right).](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d08_fit_confusion.png)
*Figure 8: Underfitting, good fit and overfitting (left three panels); the confusion matrix and its metrics (right).*


| Metric | Formula | Ask yourself |
|---|---|---|
| Accuracy | (TP + TN) / all | What share of predictions is correct? (misleading when classes are imbalanced) |
| Precision | TP / (TP + FP) | When the model says "yes", how often is it right? |
| Recall | TP / (TP + FN) | Of all real "yes" cases, how many did we catch? |
| F1-score | Harmonic mean of precision and recall | Balance of both |
| ROC-AUC | Area under the ROC curve | How well does the model rank positives above negatives (0.5 = random, 1 = perfect)? |


> **Real-world example**
>
> In cancer screening, a false negative (missing a malignant tumour) is far worse than a false alarm, so recall for the malignant class matters most. In spam filtering, precision matters more, because sending a genuine email to spam is costly. We use the real Wisconsin Breast Cancer dataset (569 samples, 30 measurements).

In [ ]:
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

# Real dataset: 569 tumour samples, 30 measurements each. Target: 0 = malignant, 1 = benign
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_test, pred),
                 "precision": precision_score(y_test, pred),
                 "recall": recall_score(y_test, pred),
                 "f1": f1_score(y_test, pred),
                 "roc_auc": roc_auc_score(y_test, proba)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

best = models["Logistic Regression"]
pred = best.predict(X_test)
print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=data.target_names))

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=data.target_names, cmap="Blues")
plt.title("Logistic Regression - confusion matrix")
plt.show()

## 4.6 Overfitting, cross-validation and hyper-parameter tuning

- **Underfitting** (high bias): the model is too simple and performs poorly on both train and test data.
- **Overfitting** (high variance): the model memorises noise. Training score is very high but test score is lower.
- **Fixes for overfitting**: more data, simpler models (limit tree depth), regularisation (L1/L2), dropout and early stopping (deep learning).
- **K-fold cross-validation**: split data into K parts, train K times, each time testing on a different part, and average the scores. More reliable than one split.
- **GridSearchCV** tries every combination of hyper-parameters (settings chosen before training, such as tree depth) with cross-validation and keeps the best.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# 1) Overfitting: as the tree gets deeper, train accuracy -> 100% but test accuracy stalls
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_acc.append(tree.score(X_train, y_train))
    test_acc.append(tree.score(X_test, y_test))

plt.plot(depths, train_acc, "o-", label="Train accuracy")
plt.plot(depths, test_acc, "s-", label="Test accuracy")
plt.xlabel("Tree depth (model complexity)"); plt.ylabel("Accuracy")
plt.title("Underfitting (left) vs overfitting (right)"); plt.legend()
plt.show()

# 2) 5-fold cross-validation: a more reliable estimate than a single split
rf = RandomForestClassifier(n_estimators=200, random_state=0)
scores = cross_val_score(rf, X, y, cv=5, scoring="accuracy")
print("CV accuracy per fold:", scores.round(3))
print(f"Mean = {scores.mean():.3f} +/- {scores.std():.3f}")

# 3) Hyper-parameter tuning with GridSearchCV
grid = GridSearchCV(DecisionTreeClassifier(random_state=0),
                    param_grid={"max_depth": [2, 3, 4, 5, 6, None],
                                "min_samples_leaf": [1, 5, 10]},
                    cv=5, scoring="accuracy")
grid.fit(X_train, y_train)
print("\nBest params:", grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_:.3f} | Test accuracy: {grid.score(X_test, y_test):.3f}")

## 4.7 Unsupervised learning: K-Means clustering

K-Means groups data into K clusters:

1. Pick K starting centres. 2. Assign every point to its nearest centre (Euclidean distance). 3. Move each centre to the average of its points. 4. Repeat until the centres stop moving.

Always scale features first, because K-Means uses distances. Choose K with the **elbow method** (where inertia stops dropping sharply), the **silhouette score** (higher is better), and business sense.

> **Real-world example**
>
> RFM segmentation is one of the most used analyses in retail. Customers are described by Recency (days since last order), Frequency (orders per year) and Monetary value (spend). Clusters such as "loyal high spenders", "regulars", "new customers" and "lapsed customers" receive different marketing campaigns. Notice in the output that the silhouette score prefers K = 3, but the elbow and the business need for four campaign groups support K = 4. Metrics guide the choice; the business decides.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# Real-world example: RFM customer segmentation
# Recency = days since last order, Frequency = orders per year, Monetary = yearly spend (Rs)
rng = np.random.default_rng(7)
groups = [  # (n, recency, frequency, monetary)
    (300, 10, 25, 60000),    # loyal high spenders
    (500, 40, 8, 15000),     # regular customers
    (400, 200, 2, 3000),     # lapsed / at-risk customers
    (200, 15, 4, 4000),      # new customers
]
parts = []
for n, r, f, m in groups:
    parts.append(pd.DataFrame({
        "recency": rng.normal(r, r * 0.2, n).clip(1),
        "frequency": rng.normal(f, f * 0.2, n).clip(1),
        "monetary": rng.normal(m, m * 0.2, n).clip(200),
    }))
rfm = pd.concat(parts, ignore_index=True)

X = StandardScaler().fit_transform(rfm)       # K-Means uses distances -> always scale

# Elbow + silhouette to choose K
inertias, sils = [], []
K = range(2, 9)
for k in K:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(K, inertias, "o-"); ax[0].set_title("Elbow method"); ax[0].set_xlabel("K")
ax[1].plot(K, sils, "o-", color="green"); ax[1].set_title("Silhouette score"); ax[1].set_xlabel("K")
plt.show()
print("Best K by silhouette:", list(K)[int(np.argmax(sils))])

km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)
rfm["segment"] = km.labels_
profile = rfm.groupby("segment").agg(customers=("recency", "size"),
                                     recency=("recency", "mean"),
                                     frequency=("frequency", "mean"),
                                     monetary=("monetary", "mean")).round(0)
print(profile.sort_values("monetary", ascending=False))

## 4.8 Module 4 practical: handwritten digit recognition

Logistics companies read handwritten PIN codes on parcels automatically. We train and compare four classifiers on 1,797 real 8 x 8 digit images, inspect the confusion matrix, and look at the images the best model gets wrong. This sets up Module 5, where a neural network will solve the harder 28 x 28 version.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

# PRACTICAL: recognise handwritten digits (think: reading PIN codes on shipping labels)
digits = load_digits()                       # 1,797 real images of 8x8 pixels
print("Data shape:", digits.data.shape, "-> each image is flattened to 64 numbers")

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, label in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(label); ax.axis("off")
plt.show()

X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.2, random_state=42, stratify=digits.target)

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "KNN (k=3)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC()),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = accuracy_score(y_test, model.predict(X_test))
print(pd.Series(results).sort_values(ascending=False).round(4))

best_name = max(results, key=results.get)
best = models[best_name]
pred = best.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues")
plt.title(f"{best_name} - confusion matrix")
plt.show()

wrong = np.where(pred != y_test)[0]
print(f"{best_name} misclassified {len(wrong)} of {len(y_test)} test images")
fig, axes = plt.subplots(1, min(6, len(wrong)), figsize=(10, 2), squeeze=False)
for ax, i in zip(axes[0], wrong[:6]):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"true {y_test[i]} / pred {pred[i]}", fontsize=8); ax.axis("off")
plt.show()